# Figure 8 | Llano conditional geological model

Run the cells in order from the repository root or `examples/`. Panels are saved to `examples/Figure/fig8/` at 400 dpi. The source field data and saved case results must be obtained separately. The assembled manuscript figure is not generated here.

This is the existing script-built conditional model. ERT1 and TEM were vetoed in the later Agent review; the model was not rebuilt and must not be described as Audit Agent-approved.

## Setup and saved case state

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap, BoundaryNorm
from matplotlib.collections import PolyCollection
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from pyproj import Transformer
from PIL import Image

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'examples').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from the project root or examples directory.')


ROOT = find_project_root()
CASE = ROOT / 'Result' / 'Llano-Uplift'
OUT = ROOT / 'examples' / 'Figure' / 'fig8'
OUT.mkdir(parents=True, exist_ok=True)
STATE = json.loads((CASE / 'case_state.json').read_text(encoding='utf-8'))

mpl.rcParams.update({
    'font.family': 'Arial', 'font.size': 11, 'axes.labelsize': 14,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'axes.linewidth': 1.1,
    'axes.grid': False, 'legend.frameon': False,
    'figure.facecolor': 'white', 'savefig.facecolor': 'white',
})

RHO_CMAP = mpl.colormaps['turbo'].copy()
RHO_CMAP.set_bad('white')
RHO_NORM = LogNorm(30, 2000)
RHO_TICKS = [30, 100, 1000]
VEL_CMAP = mpl.colormaps['viridis'].copy()
VEL_NORM = mpl.colors.Normalize(500, 2200)
UNIT_COLORS = ['#FFFFFF', '#CF604A', '#4C78A8', '#73A5A1', '#D8DCE1']
UNIT_CMAP = ListedColormap(UNIT_COLORS)
UNIT_NORM = BoundaryNorm(np.arange(-0.5, 5.5), UNIT_CMAP.N)

def bold_axes(ax):
    ax.xaxis.label.set_fontweight('bold')
    ax.yaxis.label.set_fontweight('bold')
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')
    ax.tick_params(width=1.2, length=5)

def save_png(fig, name):
    target = OUT / name
    fig.savefig(target, dpi=400, bbox_inches='tight', pad_inches=0.12,
                facecolor='white')
    # Optional vector copy for rendered-text QA; off unless the variable is set.
    qa_dir = __import__('os').environ.get('FIG8_QA_PDF_DIR')
    if qa_dir:
        fig.savefig(Path(qa_dir) / Path(name).with_suffix('.pdf'), bbox_inches='tight', pad_inches=0.12,
                    facecolor='white')
    plt.close(fig)
    return target

def vertical_colorbar(fig, cax, mappable, label, ticks):
    cb = fig.colorbar(mappable, cax=cax)
    cb.set_ticks(ticks)
    cb.set_ticklabels([str(int(tick)) for tick in ticks])
    cb.set_label(label, fontsize=13, fontweight='bold', labelpad=9)
    cb.ax.tick_params(width=1.2, labelsize=11)
    for tick in cb.ax.get_yticklabels():
        tick.set_fontweight('bold')
    return cb


In [ ]:
def load_mesh_run(run_id):
    folder = CASE / 'runs' / run_id
    mesh = np.load(folder / 'mesh_geometry.npz')
    result = json.loads((folder / 'result.json').read_text(encoding='utf-8'))
    return folder, mesh, result

ERT = {}
for item in STATE['ert_imports']:
    name = f"ERT{item['profile']}"
    folder, mesh, result = load_mesh_run(STATE['ert_runs'][name])
    ERT[name] = {
        'mesh': mesh, 'result': result,
        'electrodes': np.load(ROOT / item['electrodes_path']),
        'values': np.load(folder / 'recovered_model.npy'),
        'coverage': np.load(folder / 'model_coverage.npy'),
    }

srt_item = STATE['srt_imports'][0]
assert srt_item['profile'] == 2 and list(STATE['srt_runs']) == ['SRT2']
srt_folder, srt_mesh, srt_result = load_mesh_run(STATE['srt_runs']['SRT2'])
SRT2 = {
    'mesh': srt_mesh, 'result': srt_result,
    'sensors': np.load(ROOT / srt_item['sensors_path']),
    'values': np.load(srt_folder / 'recovered_model.npy'),
    'touched': np.load(srt_folder / 'cells_touched_by_a_ray.npy').astype(bool),
}

tdem_folder = CASE / 'runs' / STATE['tdem_run']
tdem_result = json.loads((tdem_folder / 'result.json').read_text(encoding='utf-8'))
tdem_top = np.load(tdem_folder / 'layer_top_depth_m.npy')
tdem_rho = 1.0 / np.load(tdem_folder / 'recovered_conductivity_s_m.npy')
tdem_xy = np.asarray(STATE['tdem_import']['location_m'][:2], float)

geo = np.load(CASE / 'imports' / 'geological_model.npz')
xyz = np.asarray(geo['cell_centres_m'], float)
gx, gy = [np.unique(xyz[:, i]) for i in range(2)]
nz = len(xyz) // (len(gx) * len(gy))
assert (len(gx), len(gy), nz) == (40, 40, 40)
unit = np.asarray(geo['unit'], int).reshape(40, 40, 40)
assert set(np.unique(unit)) == {0, 1, 2, 3, 4}
print('Model class counts:', dict(zip(*np.unique(unit, return_counts=True))))
print('Final run misfits:', {k: round(v['result']['summary']['chi_squared'], 2) for k, v in ERT.items()},
      'SRT2', round(srt_result['summary']['chi_squared'], 2),
      'TDEM', round(tdem_result['summary']['chi_squared'], 2))


## Survey geometry and model profile helpers

In [ ]:
from scipy.interpolate import griddata
from matplotlib.lines import Line2D
from matplotlib.patches import Circle, Rectangle
import matplotlib.patheffects as pe

utm = Transformer.from_crs('EPSG:4269', 'EPSG:26914', always_xy=True)

def survey_lines(path, id_column):
    table = np.genfromtxt(path, delimiter='\t', names=True, encoding='utf-8')
    lines = {}
    for profile in np.unique(table[id_column]):
        rows = table[table[id_column] == profile]
        distance, first = np.unique(rows['Profile_x_Coordinate_meters'], return_index=True)
        x, y = utm.transform(rows['Longitude'][first], rows['Latitude'][first])
        lines[int(profile)] = {'distance': distance, 'xy': np.c_[x, y]}
    return lines

ERT_LINES = survey_lines(ROOT / 'data' / 'case3_Llano' / 'ert' / 'electrodes_xyz.txt', 'ERT_Profile_ID')
SRT_LINES = survey_lines(ROOT / 'data' / 'case3_Llano' / 'srt' / 'geometry.txt', 'SRT_Profile_ID')

def along(line, point):
    # Distance along a surveyed line of its nearest point to `point`, and how far off it.
    s = np.linspace(line['distance'][0], line['distance'][-1], 2001)
    xy = np.c_[np.interp(s, line['distance'], line['xy'][:, 0]), np.interp(s, line['distance'], line['xy'][:, 1])]
    k = np.argmin(np.linalg.norm(xy - point, axis=1))
    return float(s[k]), float(np.linalg.norm(xy[k] - point))

def crossing(first, second):
    s1 = np.linspace(first['distance'][0], first['distance'][-1], 1501)
    s2 = np.linspace(second['distance'][0], second['distance'][-1], 1501)
    p1 = np.c_[np.interp(s1, first['distance'], first['xy'][:, 0]), np.interp(s1, first['distance'], first['xy'][:, 1])]
    p2 = np.c_[np.interp(s2, second['distance'], second['xy'][:, 0]), np.interp(s2, second['distance'], second['xy'][:, 1])]
    d = np.linalg.norm(p1[:, None] - p2[None], axis=2)
    i, j = np.unravel_index(np.argmin(d), d.shape)
    return float(s1[i]), float(s2[j])

COLUMN_ON_ERT2, _ = along(ERT_LINES[2], tdem_xy)
COLUMN_ON_SRT2, _ = along(SRT_LINES[2], tdem_xy)
ERT2_MEETS_SRT2, SRT2_MEETS_ERT2 = crossing(ERT_LINES[2], SRT_LINES[2])
ERT2_MEETS_ERT1, _ = crossing(ERT_LINES[2], ERT_LINES[1])
_, SRT2_MEETS_ERT1 = crossing(ERT_LINES[1], SRT_LINES[2])
print(f'column: ERT2 x = {COLUMN_ON_ERT2:.1f} m, SRT2 x = {COLUMN_ON_SRT2:.1f} m; '
      f'SRT2 crosses ERT2 at {SRT2_MEETS_ERT2:.1f} m and ERT1 at {SRT2_MEETS_ERT1:.1f} m')

COVERAGE_BAR_DECADES = 2.0
ERT_COLOR, EM_COLOR, SRT_COLOR = '#C8553D', '#2A5C8A', '#6F4E7C'

def reliable_ert(name):
    item = ERT[name]
    return (item['coverage'] > np.nanmax(item['coverage']) - COVERAGE_BAR_DECADES) & (item['values'] > 0)

def section_image(centres, values, keep, surface_x, surface_z, xlim, zlim, log=True, step=0.25):
    # Display interpolation of a mesh model, blanked where the model is not kept and above ground.
    xg = np.arange(xlim[0], xlim[1] + step / 2, step)
    zg = np.arange(zlim[0], zlim[1] + step / 2, step)
    X, Z = np.meshgrid(xg, zg)
    v = np.log10(values) if log else np.asarray(values, float)
    linear = griddata(centres, v, (X, Z), method='linear')
    kept = griddata(centres, keep.astype(float), (X, Z), method='nearest') > 0.5
    ground = np.interp(xg, surface_x, surface_z)
    image = np.where(np.isfinite(linear) & kept & (Z <= ground[None, :]), linear, np.nan)
    return xg, zg, (10.0 ** image) if log else image

def column_ring(ax, x, z):
    ax.scatter([x], [z], s=58, facecolor='none', edgecolor='white', linewidth=0.8, zorder=11, clip_on=False)
    ax.scatter([x], [z], s=34, facecolor='none', edgecolor='#111111', linewidth=1.1, zorder=12, clip_on=False)

def crossing_tick(ax, x, z, label, ha):
    ax.plot([x, x], [z, z + 1.6], color='#111111', lw=1.2, zorder=10, clip_on=False)
    ax.text(x + (0.8 if ha == 'left' else -0.8), z + 1.9, label, ha=ha, va='bottom',
            fontsize=10, fontweight='bold', zorder=10, clip_on=False)

def draw_ert_section(name, filename, marks, brackets=()):
    item = ERT[name]
    centres = np.asarray(item['mesh']['cell_centers_m'], float)[:, :2]
    electrodes = item['electrodes']
    xg, zg, image = section_image(centres, item['values'], reliable_ert(name),
                                  electrodes[:, 0], electrodes[:, 2], (0, 188), (267, 300))
    fig = plt.figure(figsize=(9.2, 2.7))
    ax = fig.add_axes([0.10, 0.22, 0.70, 0.56])
    cax = fig.add_axes([0.83, 0.22, 0.028, 0.56])
    mesh = ax.pcolormesh(xg, zg, image, cmap=RHO_CMAP, norm=RHO_NORM, shading='nearest', rasterized=True)
    ax.plot(electrodes[:, 0], electrodes[:, 2], color='#161B22', lw=1.5, zorder=5)
    surface = lambda x: float(np.interp(x, electrodes[:, 0], electrodes[:, 2]))
    for x, label, ha in marks:
        if label == 'column':
            column_ring(ax, x, surface(x))
        else:
            crossing_tick(ax, x, surface(x), label, ha)
    for x0, x1 in brackets:
        top = max(surface(x0), surface(x1)) + 1.2
        ax.plot([x0, x0, x1, x1], [top - 0.8, top, top, top - 0.8], color='#111111', lw=1.2, clip_on=False)
    ax.set_xlim(0, 188); ax.set_ylim(267, 300)
    ax.set_xlabel('Distance (m)'); ax.set_ylabel('Elevation (m)')
    ax.xaxis.set_major_locator(MultipleLocator(40)); ax.yaxis.set_major_locator(MultipleLocator(10))
    bold_axes(ax)
    vertical_colorbar(fig, cax, mesh, r'Resistivity ($\Omega$ m)', RHO_TICKS)
    return save_png(fig, filename)

In [ ]:
srt_centres = np.asarray(SRT2['mesh']['cell_centers_m'], float)[:, :2]
sensors = SRT2['sensors']
COLUMN_HALF_WIDTH_M = 2.5
depth_bins = np.arange(0.0, 40.01, 1.0)
depth_mid = (depth_bins[:-1] + depth_bins[1:]) / 2

def column_profile(centres, values, keep, surface_x, surface_z, at):
    near = keep & (np.abs(centres[:, 0] - at) <= COLUMN_HALF_WIDTH_M)
    depth = np.interp(centres[:, 0], surface_x, surface_z) - centres[:, 1]
    medians = np.full(depth_mid.size, np.nan)
    for i in range(depth_mid.size):
        take = near & (depth >= depth_bins[i]) & (depth < depth_bins[i + 1])
        if take.any():
            medians[i] = np.median(values[take])
    filled = np.flatnonzero(np.isfinite(medians))
    reach = float(depth_bins[filled[-1] + 1]) if filled.size else 0.0   # the bottom of the last band with data
    return medians, reach

ert2 = ERT['ERT2']
ert_profile, ert_reach = column_profile(np.asarray(ert2['mesh']['cell_centers_m'], float)[:, :2],
                                        ert2['values'], reliable_ert('ERT2'),
                                        ert2['electrodes'][:, 0], ert2['electrodes'][:, 2], COLUMN_ON_ERT2)
srt_profile, srt_reach = column_profile(srt_centres, SRT2['values'], SRT2['touched'] & np.isfinite(SRT2['values']),
                                        sensors[:, 0], sensors[:, 2], COLUMN_ON_SRT2)
shallow = tdem_top < 40.0
tdem_depth = np.r_[tdem_top[shallow], 40.0]
tdem_values = np.r_[tdem_rho[shallow], tdem_rho[shallow][-1]]
print(f'ERT2 reaches {ert_reach:.1f} m and SRT2 {srt_reach:.1f} m at the column')


## b | Model slices and coverage shares

In [ ]:
def cell_edges(centers):
    centers = np.asarray(centers, float)
    step = float(np.median(np.diff(centers)))
    return np.r_[centers[0] - step / 2, centers + step / 2]

xe = cell_edges(gx) / 1000
ye = cell_edges(gy) / 1000
legend_classes = [
    Patch(facecolor=UNIT_COLORS[2], label=r'ERT-based $<800$ $\Omega$ m'),
    Patch(facecolor=UNIT_COLORS[3], label=r'TEM-based $<800$ $\Omega$ m'),
    Patch(facecolor=UNIT_COLORS[1], label=r'TEM-based $\geq800$ $\Omega$ m'),
    Patch(facecolor=UNIT_COLORS[4], label='Low coverage'),
]
# Survey geometry is drawn as in manuscript Fig. 2 (same colours, line styles and loop), so it needs no key here.
FIG2_ERT, FIG2_SRT, FIG2_TDEM = '#D1495B', '#6F4E7C', '#E17C05'
sp_table = np.genfromtxt(ROOT / 'data' / 'case3_Llano' / 'sp' / 'survey.txt', delimiter='	', names=True, encoding='utf-8')
to_fig2 = Transformer.from_crs(4326, 26914, always_xy=True)
sp_xy = np.c_[to_fig2.transform(sp_table['Longitude'], sp_table['Latitude'])]
sp_local = np.c_[sp_table['Line_x_Coordinate_meters'], sp_table['Line_y_Coordinate_meters'], np.ones(len(sp_table))]
loop_affine, *_ = np.linalg.lstsq(sp_local, sp_xy, rcond=None)
# the 100 by 100 m loop; the 50 m TDEM model radius is the circle inscribed in it
tdem_loop = np.c_[[0.0, 100.0, 100.0, 0.0], [0.0, 0.0, 100.0, 100.0], np.ones(4)] @ loop_affine

def fig2_geometry(ax):
    for profile in (1, 2):
        xy = ERT_LINES[profile]['xy'] / 1000
        ax.plot(xy[:, 0], xy[:, 1], color='white', lw=3.8, solid_capstyle='round', zorder=4)
        ax.plot(xy[:, 0], xy[:, 1], color=FIG2_ERT, lw=1.9, solid_capstyle='round', zorder=5)
    srt_xy = SRT_LINES[2]['xy'] / 1000
    ax.plot(srt_xy[:, 0], srt_xy[:, 1], color='white', lw=3.2, zorder=4)
    ax.plot(srt_xy[:, 0], srt_xy[:, 1], color=FIG2_SRT, lw=1.6, ls=(0, (3, 1.5)), zorder=5)
    for colour, width, order in (('white', 3.6, 6), (FIG2_TDEM, 1.9, 7)):
        ax.add_patch(mpl.patches.Polygon(tdem_loop / 1000, closed=True, fill=False, edgecolor=colour,
                                         linewidth=width, zorder=order))
    column_ring(ax, tdem_xy[0] / 1000, tdem_xy[1] / 1000)

fig = plt.figure(figsize=(10.8, 6.9))
for idx, (depth_index, depth_label) in enumerate(((9, '10 m'), (29, '30 m'))):
    ax = fig.add_axes([0.09 + idx * 0.46, 0.36, 0.36, 0.56])
    ax.pcolormesh(xe, ye, unit[:, :, depth_index].T, shading='flat',
                  cmap=UNIT_CMAP, norm=UNIT_NORM, edgecolors='none', rasterized=True)
    fig2_geometry(ax)
    ax.set_xlim(xe[0], xe[-1]); ax.set_ylim(ye[0], ye[-1])
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('Easting (km)')
    if idx == 0:
        ax.set_ylabel('Northing (km)')
    else:
        ax.tick_params(axis='y', labelleft=False)
    ax.set_xticks([559.06, 559.14, 559.22])
    ax.set_yticks([3390.84, 3390.92, 3391.00])
    ax.text(0.03, 0.97, depth_label, transform=ax.transAxes, va='top', ha='left', fontsize=12,
            fontweight='bold', bbox=dict(facecolor='white', edgecolor='none', alpha=0.93, pad=2.5))
    bold_axes(ax)

# The share of the whole volume each class takes.
order = (2, 3, 1, 4, 0)
shares = np.array([np.mean(unit == k) for k in order])
bar = fig.add_axes([0.09, 0.235, 0.82, 0.047])
start = 0.0
for k, share in zip(order, shares):
    bar.barh(0, share, left=start, color=UNIT_COLORS[k], edgecolor='#9DA6AE' if k == 0 else 'none',
             linewidth=0.8, height=1.0)
    if share >= 0.05:
        bar.text(start + share / 2, 0, f'{100 * share:.1f}%', ha='center', va='center', fontsize=11,
                 fontweight='bold', color='#111111' if k in (0, 4) else 'white')
    start += share
bar.set_xlim(0, 1); bar.set_ylim(-0.5, 0.5); bar.axis('off')
print('shares:', {k: round(100 * s, 1) for k, s in zip(order, shares)})

fig.legend(handles=legend_classes, loc='lower center', bbox_to_anchor=(0.50, 0.13),
           ncol=4, fontsize=10, handlelength=2.0, columnspacing=1.15, labelspacing=1.1)
save_png(fig, 'Fig8b_model_depth_maps.png')

## a | Three-dimensional model

In [ ]:
from mpl_toolkits.mplot3d import proj3d

fig = plt.figure(figsize=(9.6, 7.4), facecolor='white')
ax = fig.add_subplot(111, projection='3d')
ax.set_proj_type('ortho')
ax.view_init(elev=26, azim=-58)
ax.set_box_aspect((1.0, 1.0, 0.42))
ax.grid(False)

xe_m, ye_m = cell_edges(gx), cell_edges(gy)
unit_v = unit[:, :, ::2]
ze = np.arange(0.0, 40.01, 2.0)
filled = unit_v != 0
face = np.array([mpl.colors.to_rgba(c) for c in UNIT_COLORS])[unit_v]
Xe, Ye, Ze = np.meshgrid(xe_m / 1000, ye_m / 1000, ze, indexing='ij')
ax.voxels(Xe, Ye, Ze, filled, facecolors=face, edgecolor='none', linewidth=0, shade=False)
ax.set_xlim(xe_m[0] / 1000, xe_m[-1] / 1000); ax.set_ylim(ye_m[0] / 1000, ye_m[-1] / 1000); ax.set_zlim(40, 0)
ax.set_xlabel('Easting (km)', labelpad=20, fontweight='bold')
ax.set_ylabel('Northing (km)', labelpad=28, fontweight='bold')
ax.set_zlabel('Depth (m)', labelpad=8, fontweight='bold')
ax.set_xticks([559.08, 559.14, 559.20]); ax.set_yticks([3390.86, 3390.92, 3390.98])
ax.set_zticks([0, 10, 20, 30, 40])
ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f'{v:.2f}'))
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f'{v:.2f}'))
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.pane.fill = False
    axis.pane.set_edgecolor('#111111')
for label in ax.get_xticklabels() + ax.get_yticklabels() + ax.get_zticklabels():
    label.set_fontweight('bold')
ax.tick_params(pad=2)
fig.subplots_adjust(left=0.0, right=0.98, bottom=0.03, top=0.98)

# Survey lines on the top face, drawn in screen space so the voxels do not hide them.
fig.canvas.draw()
projection = ax.get_proj()

# matplotlib outlines only the three back panes; close the box with the vertical edge it
# leaves open at the corner that projects furthest left, drawn like the other edges.
box_x, box_y = (xe_m[0] / 1000, xe_m[-1] / 1000), (ye_m[0] / 1000, ye_m[-1] / 1000)
corner = min(((x, y) for x in box_x for y in box_y),
             key=lambda c: proj3d.proj_transform(c[0], c[1], 0.0, projection)[0])
ex, ey, _ = proj3d.proj_transform(np.full(2, corner[0]), np.full(2, corner[1]), np.array([0.0, 40.0]), projection)
exy = fig.transFigure.inverted().transform(ax.transData.transform(np.column_stack([ex, ey])))
fig.add_artist(Line2D(exy[:, 0], exy[:, 1], transform=fig.transFigure, color=ax.xaxis.pane.get_edgecolor(),
                      lw=ax.xaxis.pane.get_linewidth(), zorder=5))
def on_top(xy, colour, width, style='-'):
    inside = ((xy[:, 0] >= xe_m[0]) & (xy[:, 0] <= xe_m[-1]) & (xy[:, 1] >= ye_m[0]) & (xy[:, 1] <= ye_m[-1]))
    xy = xy[inside] / 1000
    px, py, _ = proj3d.proj_transform(xy[:, 0], xy[:, 1], np.zeros(len(xy)), projection)
    fxy = fig.transFigure.inverted().transform(ax.transData.transform(np.column_stack([px, py])))
    fig.add_artist(Line2D(fxy[:, 0], fxy[:, 1], transform=fig.transFigure, color='white', lw=width + 1.8, zorder=999))
    fig.add_artist(Line2D(fxy[:, 0], fxy[:, 1], transform=fig.transFigure, color=colour, lw=width, ls=style, zorder=1000))
for profile in (1, 2):
    on_top(ERT_LINES[profile]['xy'], FIG2_ERT, 1.9)
on_top(SRT_LINES[2]['xy'], FIG2_SRT, 1.6, (0, (3, 1.5)))
on_top(np.vstack([tdem_loop, tdem_loop[:1]]), FIG2_TDEM, 1.9)

handles = legend_classes
fig.legend(handles=handles, loc='upper left', bbox_to_anchor=(0.02, 0.97), fontsize=10, handlelength=2.0)
save_png(fig, 'Fig8a_model_3D.png')

## Prepare borehole, published TEM comparison, and sensitivity

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'scripts'))
import case3_llano as case3
# Panel c: our TDEM model against the authors' IX1D models.
import re
text = (ROOT / 'data/case3_Llano/tdem/published_output.txt').read_text(errors='ignore')
smooth = text[text.index('Smooth Model: Occam'):]
rho_s, th_s = [], []
for line in smooth.splitlines()[5:]:
    parts = line.split()
    if not parts or not parts[0].isdigit():
        if rho_s:
            break
        continue
    rho_s.append(float(parts[1]))
    if len(parts) > 2 and re.fullmatch(r'[\d.]+', parts[2]):
        th_s.append(float(parts[2]))
# The layered model's own table: the first 'L #' header in the file.
layered = text[text.index('L #'):]
rho_l, th_l = [], []
for line in layered.splitlines()[3:]:
    parts = line.split()
    if not parts or not parts[0].isdigit():
        if rho_l:
            break
        continue
    rho_l.append(float(parts[1]))
    if len(parts) > 2:
        th_l.append(float(parts[2]))

tdem_dir = CASE / 'runs' / STATE['tdem_run']
ours_rho = 1.0 / np.load(tdem_dir / 'recovered_conductivity_s_m.npy').ravel()
ours_thick = case3._tdem_layers()

def steps(thick, rho, bottom=60.0):
    tops = np.r_[0.0, np.cumsum(thick)]
    edges = np.r_[tops, max(bottom, tops[-1] + 1.0)]
    r, z = [], []
    for value, top, base in zip(rho, edges[:-1], edges[1:]):
        if top >= bottom:
            break
        r += [value, value]; z += [top, min(base, bottom)]
    return np.array(r), np.array(z)

ours_top = np.r_[0.0, np.cumsum(ours_thick)][int(np.argmax(ours_rho >= case3.GRANITE_OHM_M))]
# Panel c: borehole D and the refraction velocity at the column.
FT = 0.3048
import csv
with open(ROOT / 'data/case3_Llano/wells/borehole_D_log.txt', encoding='utf-8') as handle:
    log = [{**row, 'Topd_Depth_feet': float(row['Topd_Depth_feet']), 'Bottom_Depth_feet': float(row['Bottom_Depth_feet'])}
           for row in csv.DictReader(handle, delimiter=chr(9))]
# Panel c: label stability with depth, per baseline unit.
sens = np.load(ROOT / STATE['model_sensitivity_path'])
geo_k = np.load(ROOT / STATE['geological_model_path'], allow_pickle=True)
unit_k = geo_k['unit']; names_k = [str(n) for n in geo_k['unit_names']]
depth_k = geo_k['ground_elevation_m'] - geo_k['cell_centres_m'][:, 2]
stable_k = (sens['unit_by_setting'] == unit_k[None, :]).all(axis=0)
bands = np.arange(0.0, 40.01, 2.0); mids = bands[:-1] + 1.0


## c | Borehole, geophysical columns, and label stability

In [ ]:
# Panel c: the comparison as one composite log at the shared column, on one depth axis.
LOG_DEPTH = 30.0
hard_rows = [(row['Topd_Depth_feet'] * FT, row['Bottom_Depth_feet'] * FT) for row in log
             if any(word in str(row['Notes']).lower() for word in ('refusal', 'very hard', 'grinding'))]
hard_top, hard_base = min(t for t, _ in hard_rows), max(b for _, b in hard_rows)
granite_top = ours_top                      # our TDEM crosses the 800 ohm m cut here; IX1D's layered model: th_l[0]

with mpl.rc_context({'font.size': 9, 'axes.labelsize': 10, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5}):
    fig = plt.figure(figsize=(7.4, 5.9))
    grid = fig.add_gridspec(1, 4, width_ratios=[0.42, 1.0, 1.45, 1.25], wspace=0.10,
                            left=0.09, right=0.84, top=0.91, bottom=0.11)
    ax_b = fig.add_subplot(grid[0, 0])
    ax_v, ax_r, ax_s = (fig.add_subplot(grid[0, k], sharey=ax_b) for k in (1, 2, 3))

    # Borehole D, in the log's own horizons.
    for row in log:
        top, base = row['Topd_Depth_feet'] * FT, row['Bottom_Depth_feet'] * FT
        notes = str(row['Notes']).lower()
        hard = any(word in notes for word in ('refusal', 'very hard', 'grinding'))
        colour = '#C9B79C' if str(row['Horizon']).lower().startswith('top') else ('#B8864B' if hard else '#EACB7A')
        ax_b.add_patch(mpl.patches.Rectangle((0, top), 1, base - top, facecolor=colour, edgecolor='#595959', lw=0.5,
                                             hatch='///' if hard else None))
    ax_b.set_xlim(0, 1); ax_b.set_xticks([])

    # SRT2 velocity at the column.
    keep_v = np.isfinite(srt_profile) & (depth_mid <= LOG_DEPTH)
    ax_v.plot(srt_profile[keep_v], depth_mid[keep_v], color=SRT_COLOR, lw=2.0, label='SRT2')
    ax_v.set_xlim(500, 1500); ax_v.set_xticks([600, 1000, 1400]); ax_v.set_xlabel(r'$V_p$ (m/s)')
    # the borehole track is too narrow for its key; it sits where SRT2 has no rays
    ax_v.add_artist(ax_v.legend(handles=[Patch(facecolor='#C9B79C', edgecolor='#595959', lw=0.5, label='Top soil'),
                                         Patch(facecolor='#EACB7A', edgecolor='#595959', lw=0.5, label='GGA'),
                                         Patch(facecolor='#B8864B', edgecolor='#595959', lw=0.5, hatch='///',
                                               label='Hard horizon')],
                                loc='lower left', fontsize=8, handlelength=1.2, borderaxespad=0.3))
    ax_v.legend(loc='upper right', fontsize=8, handlelength=1.8, borderaxespad=0.3)

    # Resistivity: ERT2 at the column, our TDEM, and the authors' IX1D models of the same sounding.
    for thick, rho, colour, ls, name in ((ours_thick, ours_rho, EM_COLOR, '-', 'TEM'),
                                          (th_l, rho_l, '#C55A11', ':', 'IX1D layered'),
                                          (th_s, rho_s, '#7F7F7F', '--', 'IX1D smooth')):
        r, z = steps(np.asarray(thick), np.asarray(rho), bottom=LOG_DEPTH)
        ax_r.plot(r, z, ls=ls, color=colour, lw=1.8, label=name)
    keep_r = np.isfinite(ert_profile) & (depth_mid <= LOG_DEPTH)
    ax_r.plot(ert_profile[keep_r], depth_mid[keep_r], color=ERT_COLOR, lw=2.0, label='ERT2')
    ax_r.axvline(case3.GRANITE_OHM_M, color='#595959', lw=0.8, ls='-.', label=r'800 $\Omega$ m')
    ax_r.set_xscale('log'); ax_r.set_xlim(10, 2e4); ax_r.set_xticks([1e2, 1e3, 1e4]); ax_r.set_xlabel(r'$\rho$ ($\Omega$ m)')
    ax_r.legend(loc='lower left', fontsize=8, handlelength=1.8, borderaxespad=0.3)

    # Label stability of the model, per baseline unit.
    for index, name in enumerate(names_k):
        if name.startswith('nothing') or name.startswith('in the volume'):
            continue
        share = [stable_k[(unit_k == index) & (depth_k >= t) & (depth_k < b)].mean()
                 if ((unit_k == index) & (depth_k >= t) & (depth_k < b)).sum() >= 20 else np.nan
                 for t, b in zip(bands[:-1], bands[1:])]
        ax_s.plot(share, mids, color=UNIT_COLORS[index], lw=2.0, label=name.split(',')[0].replace('Granite Gravel Aquifer', 'GGA'))
    ax_s.set_xlim(0, 1.02); ax_s.set_xticks([0, 0.5, 1]); ax_s.set_xticklabels(['0', '0.5', '1'])
    ax_s.legend(loc='center right', bbox_to_anchor=(1.0, 1 - 13.3 / LOG_DEPTH), fontsize=7.8, handlelength=1.2,
                borderaxespad=0.3)   # where both GGA curves sit at zero, clear of the granite curve

    # One depth axis, and the two horizons carried across every track.
    for ax, title in zip((ax_b, ax_v, ax_r, ax_s), ('Borehole D', 'Velocity', 'Resistivity', 'Label stability')):
        ax.axhspan(hard_top, hard_base, color='#B8864B', alpha=0.18, lw=0, zorder=0)
        ax.axhline(granite_top, color='#595959', lw=1.0, ls=(0, (5, 3)), zorder=1)
        ax.set_title(title, fontsize=10, fontweight='bold', pad=6)
        bold_axes(ax); ax.tick_params(width=1.0, length=3.5)
        if ax is not ax_b:
            plt.setp(ax.get_yticklabels(), visible=False)
    ax_b.set_ylim(LOG_DEPTH, 0); ax_b.yaxis.set_major_locator(MultipleLocator(5)); ax_b.set_ylabel('Depth (m)')
    ax_b.yaxis.label.set_fontweight('bold')
    edge = mpl.transforms.blended_transform_factory(ax_s.transAxes, ax_s.transData)
    ax_s.text(1.04, (hard_top + hard_base) / 2, 'Hard horizon', transform=edge, va='center', fontsize=8.5, color='#8A5A22')
    ax_s.text(1.04, granite_top, 'Top of granite', transform=edge, va='center', fontsize=8.5, color='#404040')
    save_png(fig, 'Fig8c_composite_log.png')
print(f'hard horizon {hard_top:.2f}-{hard_base:.2f} m; top of granite {granite_top:.1f} m (IX1D layered {th_l[0]:.1f} m); '
      f'ERT2 to {ert_reach:.0f} m, SRT2 to {srt_reach:.0f} m')

## Export check

Every listed panel must exist and have nonzero dimensions.

In [ ]:
from PIL import Image
expected = ['Fig8a_model_3D.png', 'Fig8b_model_depth_maps.png', 'Fig8c_composite_log.png']
for name in expected:
    path = OUT / name
    if not path.is_file():
        raise FileNotFoundError(path)
    with Image.open(path) as panel:
        assert panel.width > 0 and panel.height > 0, path
print(f'Figure panels checked: {len(expected)} in {OUT}')
